# Search-R1: Teaching a Model to Answer by Searching (end-to-end)

A guided, runnable tour of a **real Titan-RL task**: train a model to answer
open-domain questions by **calling a search tool over multiple turns**, optimized
with GRPO.

We follow the pipeline in the order you'd actually build it:

> **data -> choose a model -> define the environment (the task) -> generator vs trainer -> reward -> the loop**

Every step uses the **real Titan-RL APIs** (imported from
`torchtitan.experiments.rl`, not redefined). It runs on CPU with only `torch` +
the stdlib -- the setup cell handles imports whether or not you have the full
vLLM/GPU stack. Run the cells top to bottom and read the outputs.


### Setup
Makes the local `torchtitan` importable. In a full RL/CI env the real deps load
directly; otherwise a tiny import shim lets the pure-Python APIs load without
vLLM/Monarch/etc. Either way the classes below are the **real** ones.

In [1]:
import os, sys

def _find_repo_root(start):
    d = start
    for _ in range(8):
        if os.path.isdir(os.path.join(d, "torchtitan", "experiments", "rl")):
            return d
        nd = os.path.dirname(d)
        if nd == d: break
        d = nd
    return None

_root = _find_repo_root(os.path.abspath(os.getcwd()))
if _root and _root not in sys.path:
    sys.path.insert(0, _root)

def _install_import_shim():
    import importlib.abc, importlib.machinery, logging, types
    ROOTS = ("vllm","renderers","monarch","torchstore","cloudpickle","aiohttp","datasets")
    class _Meta(type):
        def __getattr__(cls,n): return _D
        def __call__(cls,*a,**k):
            if len(a)==1 and not k and (isinstance(a[0],type) or callable(a[0])): return a[0]
            def deco(t=None): return t if t is not None else _D
            return deco
    class _D(metaclass=_Meta):
        def __init__(self,*a,**k): pass
        def __getattr__(self,n): return _D
        def __call__(self,*a,**k): return _D
    class _Mod(types.ModuleType):
        def __getattr__(self,n):
            if n.startswith("__") and n.endswith("__"): raise AttributeError(n)
            return _D
    class _F(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        def find_spec(self,name,path=None,target=None):
            return importlib.machinery.ModuleSpec(name,self,is_package=True) if name.split(".")[0] in ROOTS else None
        def create_module(self,spec):
            m=_Mod(spec.name); m.__path__=[]; return m
        def exec_module(self,module):
            if module.__name__=="vllm.logger": module.init_logger=lambda *a,**k: logging.getLogger("stub")
            elif module.__name__=="renderers": module.Message=dict
    sys.meta_path.insert(0,_F())

try:
    from torchtitan.experiments.rl.rollout.types import RolloutStatus
    MODE="full environment (real deps)"
except Exception:
    _install_import_shim()
    from torchtitan.experiments.rl.rollout.types import RolloutStatus
    MODE="lightweight (import shim active)"
print("Titan-RL ready:", MODE)


Titan-RL ready: lightweight (import shim active)


## Step 1 - The task & the data

**Task:** open-domain question answering where the model may **search** a
knowledge base before answering (Search-R1). Each training example is one
question plus its accepted answers.

The real type is `SearchR1Sample`. In a real run, `SearchR1Dataset` streams these
from the HF dataset `PeterJinGo/nq_hotpotqa_train` (NQ + HotpotQA). Here we just
build one by hand so you can see its shape.

In [2]:
from torchtitan.experiments.rl.examples.search_r1.data import SearchR1Sample

sample = SearchR1Sample(
    question="Who wrote the play Hamlet?",
    golden_answers=["Shakespeare", "William Shakespeare"],   # any match counts (exact-match)
)
print("question      :", sample.question)
print("golden answers:", sample.golden_answers)
# (real run: SearchR1Dataset.Config(filename='train.parquet').build() yields an endless stream of these)


question      : Who wrote the play Hamlet?
golden answers: ['Shakespeare', 'William Shakespeare']


## Step 2 - Choose your model

Model selection lives in the **config registry**
(`examples/search_r1/config_registry.py`). Two ready recipes:

| Config | Model | GPU split (8 GPUs) |
|---|---|---|
| `rl_grpo_qwen3_1_7b_search_r1` | Qwen3-1.7B | 4 generator (TP=4) + 1 trainer + retriever |
| `rl_grpo_qwen3_8b_search_r1`   | Qwen3-8B   | 2 generator (TP=2) + 4 trainer (TP=4) + retriever |

You pick the model with one line -- `model_registry("1.7B" | "8B", ...)` -- and
the recipe fills everything else (GRPO, DAPO loss, optimizer, parallelism). Let's
build the 1.7B model spec and peek at it.

In [3]:
from torchtitan.models.qwen3 import model_registry

model_spec = model_registry("1.7B", attn_backend="varlen")   # <-- swap "1.7B" -> "8B" to scale up
print("model spec  :", type(model_spec).__name__)
print("max_seq_len :", model_spec.model.max_seq_len)
print("num layers  :", len(model_spec.model.layers))
print("\nThis one ModelSpec is used by BOTH the trainer and (registered into) the generator.")
print("Recipes available: rl_grpo_qwen3_1_7b_search_r1 , rl_grpo_qwen3_8b_search_r1")


model spec  : ModelSpec
max_seq_len : 40960
num layers  : 28

This one ModelSpec is used by BOTH the trainer and (registered into) the generator.
Recipes available: rl_grpo_qwen3_1_7b_search_r1 , rl_grpo_qwen3_8b_search_r1


## Step 3 - Define the environment (the task logic)

The environment is where **you** describe the task, in messages. `SearchR1Env`
(a `MessageEnv`) does two things:
- `init()` -> asks the question and exposes a **`search` tool**.
- `step(answer)` -> if the model **called search**, run it and return the passages
  (another turn); if the model **just answered**, end the rollout.

That "call search -> get passages -> maybe search again -> answer" is the
**multi-turn** loop. Below we build the env and drive it with a fake retrieval
function (a stand-in for the real search server) so it runs offline.

In [4]:
import asyncio, types
from torchtitan.experiments.rl.examples.search_r1 import env as sr1

print("tool exposed to the model:", sr1.SEARCH_TOOL["name"], "-", sr1.SEARCH_TOOL["description"][:60], "...")

# stand-in for the real dense-retrieval server (no network here)
async def fake_search(query, *, url, topk, timeout_s=60.0):
    return f"Doc 1(Title: Hamlet) Hamlet is a tragedy written by William Shakespeare."
sr1._search = fake_search

env = sr1.SearchR1Env(sr1.SearchR1Env.Config(), env_input=sample)

async def run_episode():
    step = await env.init()
    print("\n[turn 0] env asks:", step.init_prompt_messages[0]["content"][:70], "...")

    # turn 1: the model decides to search
    tool_call = types.SimpleNamespace(name="search", arguments={"query": "who wrote Hamlet"})
    step = await env.step({"role": "assistant", "content": "", "tool_calls": [tool_call]})
    print("[turn 1] model called search -> env replies with a tool message:")
    print("         ", step.env_messages[0]["content"])
    print("         rollout done?", step.done, "(no -> another turn)")

    # turn 2: the model now answers (no tool call) -> episode ends
    step = await env.step({"role": "assistant", "content": "Shakespeare"})
    print("[turn 2] model answered 'Shakespeare' -> rollout done?", step.done)

await run_episode()


tool exposed to the model: search - Search a Wikipedia-derived knowledge base and return the top ...

[turn 0] env asks: Answer the question. Use the search tool to look up any facts you are  ...
[turn 1] model called search -> env replies with a tool message:
          Doc 1(Title: Hamlet) Hamlet is a tragedy written by William Shakespeare.
         rollout done? False (no -> another turn)
[turn 2] model answered 'Shakespeare' -> rollout done? True


## Step 3b - Bring your own environment

Swapping the task = writing your **own `MessageEnv`** (the *only* piece you rewrite - generation, GRPO, weight sync stay the same). Same two methods: `init()` sets the first prompt, `step()` decides **done vs another turn**, and can branch however you like (no tool required). Below is a tiny custom env with a one-time 'hint' follow-up - it plugs into the exact same rollout loop as Search-R1.


In [5]:
from dataclasses import dataclass
from torchtitan.experiments.rl.environment import (
    MessageEnv, MessageEnvInitOutput, MessageEnvStepOutput,
)

class MyQuizEnv(MessageEnv):
    @dataclass(kw_only=True, slots=True)
    class Config(MessageEnv.Config):
        question: str = "What is the capital of France?"

    def __init__(self, config, *, env_input=None):
        self._q = config.question
        self._gave_hint = False

    async def init(self):                              # first prompt the model sees
        return MessageEnvInitOutput(
            init_prompt_messages=[{"role": "user",
                                   "content": self._q + "  (say 'hint' if unsure)"}])

    async def step(self, completion_message):          # your custom turn logic
        said = (completion_message.get("content") or "").lower()
        if "hint" in said and not self._gave_hint:     # branch: give one hint, keep going
            self._gave_hint = True
            return MessageEnvStepOutput(
                env_messages=[{"role": "user", "content": "Hint: it is on the Seine."}],
                done=False)
        return MessageEnvStepOutput(done=True, env_rewards={"answered": 1.0})   # else finish

# drive it exactly like Search-R1 (canned model replies)
async def run_quiz():
    env = MyQuizEnv.Config().build(env_input=None)
    print("[turn 0] env asks:", (await env.init()).init_prompt_messages[0]["content"])
    s1 = await env.step({"role": "assistant", "content": "hint"})
    print("[turn 1] model said 'hint' -> done?", s1.done, "| env replies:", s1.env_messages[0]["content"])
    s2 = await env.step({"role": "assistant", "content": "Paris"})
    print("[turn 2] model answered 'Paris' -> done?", s2.done, "| env_rewards:", s2.env_rewards)

await run_quiz()
# To use it in a real run: put MyQuizEnv.Config() as `message_env` in a Rollouter config,
# pair it with a RewardFn, and everything else is unchanged.


[turn 0] env asks: What is the capital of France?  (say 'hint' if unsure)
[turn 1] model said 'hint' -> done? False | env replies: Hint: it is on the Seine.
[turn 2] model answered 'Paris' -> done? True | env_rewards: {'answered': 1.0}


## Step 4 - Generator vs Trainer: two copies, two jobs

The same model is run two ways, on two GPU pools:

| | **Generator** (`VLLMGenerator`) | **Trainer** (`PolicyTrainer`) |
|---|---|---|
| job | sample answers fast (rollouts) | learn from them (gradient update) |
| engine | vLLM (KV cache, continuous batching) | TorchTitan (FSDP/TP, optimizer, backprop) |
| 1.7B recipe | TP=4, bf16, cudagraph on | TP=1, AdamW lr=1e-6 |

Below: the real generator config (what samples the answers) and the real loss the
trainer optimizes (DAPO clip-higher). The generator produces the `Completion`s and
their logprobs; the trainer recomputes logprobs and applies the GRPO update.

In [6]:
from torchtitan.experiments.rl.actors.generator import VLLMGenerator, SamplingConfig
from torchtitan.experiments.rl.models.vllm_registry import InferenceParallelismConfig
from torchtitan.experiments.rl.losses import DAPOLoss

# --- GENERATOR: how answers are sampled ---
gen_cfg = VLLMGenerator.Config(
    model_dtype="bfloat16",
    parallelism=InferenceParallelismConfig(tensor_parallel_degree=4),
    sampling=SamplingConfig(temperature=1.0, top_p=1.0, max_tokens=512),
)
print("GENERATOR:  dtype=", gen_cfg.model_dtype, "| TP=", gen_cfg.parallelism.tensor_parallel_degree,
      "| sampling=", gen_cfg.sampling)

# --- TRAINER: the objective it optimizes (DAPO clip-higher, no KL/ref model) ---
loss_cfg = DAPOLoss.Config(ratio_clip_low=0.2, ratio_clip_high=0.28)
print("TRAINER loss:", type(loss_cfg).__name__, "| clip_low=", loss_cfg.ratio_clip_low,
      "| clip_high=", loss_cfg.ratio_clip_high, " (asymmetric = 'clip-higher')")
print("\nSame model, two runtimes: generator samples, trainer learns; weights sync generator <- trainer each step.")


initial_load_model_only=True has no effect without an initial_load_path.


GENERATOR:  dtype= bfloat16 | TP= 4 | sampling= SamplingConfig(temperature=1.0, top_p=1.0, max_tokens=512, seed=None, stop_token_ids=None)
TRAINER loss: Config | clip_low= 0.2 | clip_high= 0.28  (asymmetric = 'clip-higher')

Same model, two runtimes: generator samples, trainer learns; weights sync generator <- trainer each step.


## Step 5 - The reward: did it answer correctly?

Scoring is a `RewardFn` inside a `Rubric`. Search-R1 uses `RewardExactMatch`:
the model's **final answer** is normalized and compared to the golden answers
(1.0 if it matches, else 0.0). Optional levers put *searching* on the gradient
(`no_search_penalty`, `retrieval_score`). Let's score a correct rollout and a
wrong one.

In [7]:
from torchtitan.experiments.rl.examples.search_r1.rubric import RewardExactMatch
from torchtitan.experiments.rl.rubrics import Rubric
from torchtitan.experiments.rl.rollout.types import Rollout, RolloutTurn, RolloutStatus
from torchtitan.experiments.rl.types import RolloutTurnID

def make_rollout(final_answer):
    # turn 0 = a search tool call; turn 1 = the final answer (what gets graded)
    searched = RolloutTurn(rollout_id=RolloutTurnID(0,0,0), prompt_token_ids=[1],
                           completion_token_ids=[2], completion_logprobs=[-0.1],
                           completion_message={"tool_calls":[1]})
    answered = RolloutTurn(rollout_id=RolloutTurnID(0,0,1), prompt_token_ids=[1],
                           completion_token_ids=[2], completion_logprobs=[-0.1],
                           completion_message={"content": final_answer})
    return Rollout(group_id=0, rollout_id=0, status=RolloutStatus.COMPLETED, turns=[searched, answered])

rubric = Rubric.Config(reward_fns=[RewardExactMatch.Config(weight=1.0)], truncation_reward=0.0).build()
outs = await rubric.score_group([make_rollout("Shakespeare"), make_rollout("Francis Bacon")], sample)
print("answer 'Shakespeare'  -> reward", outs[0].reward)
print("answer 'Francis Bacon'-> reward", outs[1].reward)


answer 'Shakespeare'  -> reward 1.0
answer 'Francis Bacon'-> reward 0.0


## Step 6 - Rewards -> advantages -> policy update

GRPO samples a **group** of answers per question and grades on the group's own
average: `advantage = reward - mean(reward)`. Positive = beat the group -> make
those tokens more likely; negative -> less. The trainer applies the DAPO clipped
loss from Step 4. Here are advantages for a group of 4 answers.

In [8]:
from torchtitan.experiments.rl.rollout.advantage import AdvantageEstimator
from torchtitan.experiments.rl.rollout.types import RolloutGroup

group = RolloutGroup(group_id=0, rollouts=[
    make_rollout(a) for a in ["Shakespeare", "Francis Bacon", "William Shakespeare", "Marlowe"]
])
for r, reward in zip(group.rollouts, [1.0, 0.0, 1.0, 0.0]):
    r.reward = reward

advantages = AdvantageEstimator.Config(should_std_normalize=True).build()(group)
print("rewards   :", [r.reward for r in group.rollouts])
print("advantages:", [round(a, 2) for a in advantages], " (correct answers > 0, wrong < 0)")


rewards   : [1.0, 0.0, 1.0, 0.0]
advantages: [1.0, -1.0, 1.0, -1.0]  (correct answers > 0, wrong < 0)


## Step 7 - Run it for real, and what to watch

Everything above is the data plane you just ran on CPU. The full run adds the GPU
actors (generator + trainer) and the retrieval server, driven by the `Controller`:

```bash
# needs: 8 GPUs, a running dense-retrieval server, and the QA parquet (see examples/search_r1/README.md)
python -m torchtitan.experiments.rl.train \
  --module search_r1 --config rl_grpo_qwen3_1_7b_search_r1
```

**What to watch:** the **validation exact-match accuracy** rising over training
(the "did it learn to search+answer?" signal). The example ships reference curves
for Qwen3-1.7B and 8B in `examples/search_r1/assets/`.

### What you saw (the pipeline, with the real APIs)
| Step | Real API | Turned ... into ... |
|---|---|---|
| data | `SearchR1Sample` / `SearchR1Dataset` | a question + golden answers |
| model | `model_registry("1.7B")` | one ModelSpec for trainer + generator |
| environment | `SearchR1Env` (`MessageEnv`) | the multi-turn search task |
| generation | `VLLMGenerator` + `SamplingConfig` | prompt -> sampled answers |
| training | `PolicyTrainer` + `DAPOLoss` | answers -> gradient update |
| reward | `RewardExactMatch` + `Rubric` | answer -> 0/1 reward |
| advantage | `AdvantageEstimator` | group rewards -> per-answer advantage |

**Read next:** `examples/search_r1/{env,rubric,data,config_registry}.py`, then
`controller.py` for how the 4 loops drive all of this. Deeper: `docs/test_plan.md`,
`docs/rl_architecture_and_forge_comparison.md`.
